# Customer Churn Prediction - ML Pipeline

This notebook builds a machine learning pipeline to predict customer churn for a telecom company.

## 1. Setup and Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

## 2. Load and Explore Data

In [2]:
# Load the dataset
data_path = '/data/customer_churn.csv'
df = pd.read_csv(data_path)
print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")

In [3]:
# Basic exploration
print("\nFirst few rows:")
print(df.head())
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())

## 3. Data Cleaning and Preprocessing

In [4]:
# Handle missing values
df_clean = df.copy()
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col].fillna(df_clean[col].median(), inplace=True)

In [5]:
# Handle categorical missing values
categorical_cols = df_clean.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col].fillna(df_clean[col].mode()[0], inplace=True)

In [6]:
# Remove duplicates
initial_rows = len(df_clean)
df_clean = df_clean.drop_duplicates()
duplicates_removed = initial_rows - len(df_clean)
print(f"Removed {duplicates_removed} duplicate rows")

In [7]:
# Remove outliers using IQR method for numeric columns
df_no_outliers = df_clean.copy()
for col in numeric_cols:
    if col != 'Churn':  # Don't remove outliers from target
        Q1 = df_no_outliers[col].quantile(0.25)
        Q3 = df_no_outliers[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        df_no_outliers = df_no_outliers[
            (df_no_outliers[col] >= lower_bound) & 
            (df_no_outliers[col] <= upper_bound)
        ]
print(f"Dataset after outlier removal: {len(df_no_outliers)} rows")

## 4. Feature Engineering

In [8]:
# Create new features
df_features = df_no_outliers.copy()

# Create tenure groups
if 'tenure' in df_features.columns:
    df_features['tenure_group'] = pd.cut(
        df_features['tenure'], 
        bins=[0, 12, 24, 48, 72], 
        labels=['0-12', '12-24', '24-48', '48-72']
    )

# Create average monthly charge
if 'TotalCharges' in df_features.columns and 'tenure' in df_features.columns:
    df_features['avg_monthly_charge'] = df_features['TotalCharges'] / (df_features['tenure'] + 1)

In [9]:
# Create service usage score
service_cols = ['PhoneService', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport']
available_service_cols = [col for col in service_cols if col in df_features.columns]

if available_service_cols:
    df_features['service_count'] = 0
    for col in available_service_cols:
        df_features['service_count'] += (df_features[col] == 'Yes').astype(int)

In [10]:
# Create contract value indicator
if 'Contract' in df_features.columns:
    df_features['is_long_term'] = (df_features['Contract'].isin(['One year', 'Two year'])).astype(int)

# Create payment method risk score
if 'PaymentMethod' in df_features.columns:
    risky_payment = ['Electronic check']
    df_features['risky_payment'] = df_features['PaymentMethod'].isin(risky_payment).astype(int)

print(f"Feature engineering complete. Total features: {len(df_features.columns)}")

## 5. Encode Categorical Variables

In [11]:
# Encode categorical variables
df_encoded = df_features.copy()
label_encoders = {}

categorical_columns = df_encoded.select_dtypes(include=['object', 'category']).columns
categorical_columns = [col for col in categorical_columns if col != 'Churn']  # Don't encode target yet

for col in categorical_columns:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))
    label_encoders[col] = le

In [12]:
# Encode target variable
if 'Churn' in df_encoded.columns:
    target_encoder = LabelEncoder()
    df_encoded['Churn'] = target_encoder.fit_transform(df_encoded['Churn'])
    print(f"Target classes: {target_encoder.classes_}")

## 6. Prepare Training Data

In [13]:
# Separate features and target
X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Target distribution: {y.value_counts().to_dict()}")

In [14]:
# Split into train and test sets
random_state = 42
test_size = 0.2

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=test_size, 
    random_state=random_state,
    stratify=y
)

print(f"\nTraining set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

In [15]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling complete")

## 7. Train Model

In [16]:
# Initialize model
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=random_state,
    n_jobs=-1
)

print("Training Random Forest model...")

In [17]:
# Train the model
model.fit(X_train_scaled, y_train)
print("Model training complete!")

In [18]:
# Get feature importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 Most Important Features:")
print(feature_importance.head(10))

## 8. Evaluate Model

In [19]:
# Make predictions
y_train_pred = model.predict(X_train_scaled)
y_test_pred = model.predict(X_test_scaled)

# Get prediction probabilities
y_test_proba = model.predict_proba(X_test_scaled)[:, 1]

In [20]:
# Calculate metrics for training set
train_accuracy = accuracy_score(y_train, y_train_pred)
train_precision = precision_score(y_train, y_train_pred)
train_recall = recall_score(y_train, y_train_pred)
train_f1 = f1_score(y_train, y_train_pred)

print("Training Set Performance:")
print(f"Accuracy: {train_accuracy:.4f}")
print(f"Precision: {train_precision:.4f}")
print(f"Recall: {train_recall:.4f}")
print(f"F1 Score: {train_f1:.4f}")

In [21]:
# Calculate metrics for test set
test_accuracy = accuracy_score(y_test, y_test_pred)
test_precision = precision_score(y_test, y_test_pred)
test_recall = recall_score(y_test, y_test_pred)
test_f1 = f1_score(y_test, y_test_pred)

print("\nTest Set Performance:")
print(f"Accuracy: {test_accuracy:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"Recall: {test_recall:.4f}")
print(f"F1 Score: {test_f1:.4f}")

In [22]:
# Calculate confusion matrix
cm = confusion_matrix(y_test, y_test_pred)
tn, fp, fn, tp = cm.ravel()

print("\nConfusion Matrix:")
print(f"True Negatives: {tn}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"True Positives: {tp}")

## 9. Save Results

In [23]:
# Save model performance summary
results_path = '/output/model_results.csv'
results_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score'],
    'Train': [train_accuracy, train_precision, train_recall, train_f1],
    'Test': [test_accuracy, test_precision, test_recall, test_f1]
})

print("\nModel Results Summary:")
print(results_df)

In [24]:
# Save feature importance
feature_importance_path = '/output/feature_importance.csv'
print(f"\nFeature importance and results ready for export")
print(f"Model training and evaluation complete!")